# Milestone 2 Exploration  (Internal Only)

This notebook is to confirm our .py script outputs, not for submission.

In [64]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [65]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [66]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   parent_asin                    20000 non-null  str    
 1   product_title                  20000 non-null  str    
 2   features                       20000 non-null  object 
 3   description                    20000 non-null  object 
 4   categories                     20000 non-null  object 
 5   details                        20000 non-null  object 
 6   price                          12100 non-null  float64
 7   derived_avg_rating             20000 non-null  float64
 8   max_helpful_vote               20000 non-null  int64  
 9   n_reviews                      20000 non-null  int64  
 10  review_text                    20000 non-null  str    
 11  candidate_review_title         20000 non-null  str    
 12  candidate_review_text          20000 non-null  str    
 1

In [48]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,max_helpful_vote,n_reviews,review_text,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
0,B008M2IPBE,"Whirlpool 833697 Condenser, Motor Only. Does n...",[The Whirlpool Condenser Fan Motor is a genuin...,"[Product Description, The high quality Whirlpo...","[Appliances, Parts & Accessories, Refrigerator...","[(Brand, ""Whirlpool""), (Model Name, ""833697""),...",38.09,4.833333,2.0,6,Works great! Saved money: Saved me a bunch of ...,Perfect Replacement for My Older Amana Fridge,The condenser motor on our older Amana fridge ...,2.0
1,B09P2SCNZ3,12001541 303373K Dryer Drum Roller Support Whe...,"[【Premium Quality】High-quality materials, prec...",[],"[Appliances, Parts & Accessories, Dryer Parts ...","[(Manufacturer, ""Grete Gotye""), (Part Number, ...",19.99,3.153846,3.0,13,Cheap junk!!: These really looked and felt che...,Quality questionable,The metal washers supplied with the kit were t...,3.0
2,B0C57WMPJQ,"Silonn Ice Makers Countertop, 9 Cubes Ready in...",[Note : Please check the dimension and item we...,[Ice in 6 mins - Self Cleaning - Portable - 2 ...,"[Appliances, Refrigerators, Freezers & Ice Mak...","[(Brand, ""Silonn""), (Model Name, ""SLIM01B""), (...",80.97,4.044843,360.0,223,"Died of Natural Causes: Purchased 11/2021, Die...",Piece of junk...DO NOT BUY!,I bought this portable ice maker after reviewi...,360.0
3,B01MUAQ2FW,Integra Boost RH 62% 2 Way Humidity Control Me...,[Use our Integra Boost 2-way humidity control ...,[Integra’s 2-way humidity control technology r...,"[Appliances, Parts & Accessories, Humidifier P...","[(Package Dimensions, ""6 x 5.5 x 2.2 inches""),...",9.99,3.142857,2.0,7,Like having nothing at all!: Went by instructi...,Not as good as the others,"Kinda dry when they arrived, even after rehydr...",2.0
4,B095WKKHCJ,MAOPINER Multi-Functional Furniture Dolly Roll...,[Adjustable Dolly Roller Base - This movable b...,[],"[Appliances, Laundry Appliances, Washers & Dry...","[(Manufacturer, ""MAOPINER""), (Part Number, ""MP...",22.99,1.000000,0.0,1,Coloqué la lavadora y se quebraron las ruedas:...,Coloqué la lavadora y se quebraron las ruedas,No es fuerte coloqué la lavadora y se quedará ...,0.0


In [68]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
price,7900,0.395
parent_asin,0,0.000
product_title,0,0.000
features,0,0.000
description,0,0.000
categories,0,0.000
details,0,0.000
derived_avg_rating,0,0.000
max_helpful_vote,0,0.000
n_reviews,0,0.000


In [69]:
df.describe()

,price,derived_avg_rating,max_helpful_vote,n_reviews,candidate_review_helpful_vote
count,12100.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,72.725421,4.239227,4.316900,6.861950,4.316900
std,226.290880,1.120545,30.098584,29.901266,30.098584
min,1.940000,1.000000,0.000000,1.000000,0.000000
25%,14.260000,4.000000,0.000000,1.000000,0.000000
50%,24.990000,4.789474,0.000000,2.000000,0.000000
75%,49.950000,5.000000,2.000000,4.000000,2.000000
max,6999.000000,5.000000,1835.000000,1329.000000,1835.000000


## Debugging Code

The code below were explorations to help inform the pipeline and find bugs. Codex was used to generate quick debugging code below.

In [51]:
def present(s):
    return s.notna() & s.astype(str).str.strip().ne("")

df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

summary = pd.crosstab(
    index=[
        df["has_n_reviews"],
        df["has_avg_rating"],
        df["has_candidate_title"],
        df["has_candidate_text"],
    ],
    columns="count",
).reset_index()

summary = summary.sort_values("count", ascending=False)
summary

col_0,has_n_reviews,has_avg_rating,has_candidate_title,has_candidate_text,count
0,False,False,False,False,11331
2,True,True,True,True,8668
1,True,True,True,False,1


In [52]:
# Review count exists, but no candidate review shown
df.loc[
    (df["n_reviews"] > 0)
    & (~present(df["candidate_review_title"]))
    & (~present(df["candidate_review_text"])),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
].head(20)

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote


In [58]:
def present(s):
    cleaned = s.astype("string").str.strip()
    missing_tokens = {"", "na", "n/a", "nan", "none", "null"}
    return cleaned.notna() & ~cleaned.str.lower().isin(missing_tokens)

In [59]:
df["has_n_reviews"] = df["n_reviews"].fillna(0) > 0
df["has_avg_rating"] = df["derived_avg_rating"].notna()
df["has_candidate_title"] = present(df["candidate_review_title"])
df["has_candidate_text"] = present(df["candidate_review_text"])
df["has_any_candidate_review"] = df["has_candidate_title"] | df["has_candidate_text"]

In [61]:
df.loc[
    df["product_title"].str.contains(
        "Pour Over Coffee Dripper Stainless Steel Double Layer Mesh",
        regex=False,
        na=False,
    ),
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
4879,B07D6L2QX2,Pour Over Coffee Dripper Stainless Steel Doubl...,0,NaN,NaN,NaN,NaN


In [63]:
df.loc[
    df["parent_asin"] == "B09VT3BS1G",
    [
        "parent_asin",
        "product_title",
        "n_reviews",
        "derived_avg_rating",
        "candidate_review_title",
        "candidate_review_text",
        "candidate_review_helpful_vote",
    ],
]

,parent_asin,product_title,n_reviews,derived_avg_rating,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
10901,B09VT3BS1G,"MoMoSun Furniture Dolly,Extendable Washing Mac...",0,NaN,NaN,NaN,NaN
